In [ ]:
from pathlib import Path
import random 
import pandas as pd

# USER CONFIG: change these values.
MODEL_ROOT = Path("/tmp/cube555_inputs/model")
CHECKPOINT_PATH = MODEL_ROOT / "q555_f1_bell2k.pt"  # Transformer
RESMLP_CHECKPOINT_PATH = MODEL_ROOT / "q555_2k_BEST.pt"  # ResMLP

####################################################################
#PUZZLE_ID_START = 1000  # inclusive
#PUZZLE_ID_END = 1000  # inclusive
url = "https://raw.githubusercontent.com/chervov/Test01/master/list555_1.txt"
# For comma-separated or plain single-column text:
df = pd.read_csv(url, header=None)
list_indices = df.iloc[0,:].to_list()
print('Loaded indices from github:',list_indices )
rand_index = random.randint(0, len(list_indices)-1 )
PUZZLE_ID_START = list_indices[rand_index]   # inclusive
PUZZLE_ID_END = PUZZLE_ID_START  # inclusive
print('PUZZLE_ID_START',PUZZLE_ID_START,'PUZZLE_ID_END',PUZZLE_ID_END )
####################################################################

# At depth 140, do not exceed 3_100_000 for a single 12h Kaggle session.
# Larger widths risk exceeding the limit; 3_200_000 is estimated above 12h.
BEAM_WIDTH = 3_100_000  # about 11h49 to depth 140 by interpolation; no slowdown margin
#BEAM_WIDTH = 2**18# # 500_000
print('BEAM_WIDTH',BEAM_WIDTH)

MAX_DEPTH = 140
TRANSFORMER_WEIGHT = 0.8  # 0 = ResMLP, 1 = Transformer

REFLECT_MODE = "off"  # off | after_original | only
REFLECT_SOURCE_CSV = None  # path to solutions CSV; required for "only"
SOLUTION_MODE = "collect"  # first | collect
COLLECT_EXTRA_DEPTHS = 1  # collect: additional depths after the first solution (0, 1, 2, ...)
MAX_COLLECTED_SOLUTIONS = 2_000

KAGGLE_OWNER = "trydotatwo"
KAGGLE_SLUG = "cube555-native-2xt4-blend"
KAGGLE_VERSION = 62  # saved notebook version; when forking, set your owner/slug/version


# CayleyPy Cube555 - native beam on 2xT4

Artgor PieceTransformerQ555 + ResMLPQ, parent-Q blend **0.8 / 0.2**.
The two ranks share one native beam. This is the existing MultiGPUBeamSearch
pipeline with a Cube555 LibTorch scoring module, not the TPU/JAX search engine.
FP16 on T4 can choose different paths from BF16 on TPU. The reported 102/100
results have not been reproduced by this notebook.

Run All with Internet enabled. Source, model weights, layout and competition CSVs
are downloaded from GitHub; no attached Kaggle datasets or competitions are required.
The input ZIP and every extracted file are verified with SHA-256.
Enter `BEAM_WIDTH` directly. Profiles: 65536, 131072, 262144, 524288, 1048576, 2097152, 4000000.
Requested widths up to 4000000 are accepted; batch alignment may round up
(the maximum request becomes 4005888 internally). Larger requests are rejected.
The default beam is 3100000 at depth 140: approximately 11 h 49 min,
interpolated from measured 2097152 and 4005888 full-step timings.
Preparation/BFS/save are included; no slowdown or extra time margin is included.
This exact width has not been run separately, and this is not a 12-hour guarantee.
**At depth 140, do not exceed BEAM_WIDTH=3_100_000 for a single 12-hour Kaggle session.**
Larger widths risk exceeding the limit; 3_200_000 is estimated at about 12 h 12 min.
Use 2097152 for more time headroom. Width 4000000 at depth 140 exceeds 12 h.
All seven listed profiles passed bounded two-rank allocation and saturated-step checks.
At 131072 pipeline throughput was 6.0% below isolated Stream1; at 262144, 1.4%.
At 524288 and above the difference was within surrounding-control drift.
The 65536 overhead block was inconclusive (11.8% control drift).
Isolated legal-walk states and actual search frontier populations differ.
Only compatible Cube555 PieceTransformerQ555 / ResMLPQ weights are supported.
Keep layout and puzzle_info.json beside the model bundle.
Cube555 capacity is checked by the native
150/160-byte memory planner; Cube4's measured ceiling is not a Cube555 claim.
`B_MICRO=8192` is the outer parent transaction. `MODEL_MICRO` independently limits
one LibTorch forward. The original Transformer registry selects shards, Stream4
buffers and final exchange chunks from the requested width.
The default is MODEL_MICRO=128 with one inference lane. Fixed-work, equal-warmup
paired controls found no confirmed faster challenger; several small differences
remain unresolved. A paired full-step check retained the baseline pipeline;
the combined shard/batch alternative had an unstable control block.
History uses at most 24 GB RAM, preserves at least 6 GB of available RAM
for runtime overhead, and uses 50 GiB scratch disk. Default depth is 140.
Collect staging is bounded by MAX_COLLECTED_SOLUTIONS * effective MAX_DEPTH
records (40 bytes each per GPU), capped by the layer candidate count.
A layer exceeding this budget logs a warning and dropped-hit counts, saves
the stored hits, and continues search. performance.json records truncation.
The native total-memory preflight still applies to the beam and all other buffers. Preflight explicitly reports
requested/effective MAX_DEPTH; beam is never silently reduced to fit history.
All returned solutions are replayed. Both rank logs and provenance are retained.
`gpu_samples.csv` samples both GPUs every second, and `performance.json` records
memory high-water samples and native per-depth timings. Sampling can miss brief peaks.
Result publication is on by default and applies only to locally replayed solutions.
Each puzzle retains `publish_status.json` and the exact `results-*.json.gz` request.
HTTP acceptance is not a claim that the GitHub promotion has completed.
When forking, update the Kaggle owner, slug and saved version in the config.
The result author is derived from your Kaggle username; an explicit author_name
in publication metadata can override the display name.
The notebook-source hash is derived from actual running cells, not a placeholder.


In [ ]:
# Runtime settings: normally leave unchanged.
SOLVER_COMMIT = '1ee313325a39371492c043a380f0af78cd7713ff'
SMOKE_TEST = False
COMPETITION_ROOT = MODEL_ROOT.parent / "competition"
LAYOUT_PATH = MODEL_ROOT / "piece_layout_555.json"
TOUCH_BFS_RADIUS = 5
B_MICRO = 8192
MODEL_MICRO = 128
PUBLISH_RESULTS = True
PUZZLE_IDS = list(range(PUZZLE_ID_START, PUZZLE_ID_END + 1))
if not PUZZLE_IDS:
    raise ValueError("PUZZLE_ID_END must be >= PUZZLE_ID_START")
if BEAM_WIDTH >= 3_100_000 and MAX_DEPTH >= 140:
    print("Time warning: this beam/depth may exceed Kaggle's 12h session. "
          "Use a smaller beam or depth; the requested beam will not be reduced automatically.")
# Download the public, pinned GitHub inputs before checking paths.
import hashlib, json, shutil, urllib.request, zipfile
INPUT_BUNDLE_URL = "https://github.com/TryDotAtwo/MultiGPUBeamSearch/releases/download/cube555-inputs-20261001/cube555-inputs.zip"
INPUT_BUNDLE_SHA256 = "49d61a127540cd197db9bf75aacad5712a7ab54d3fbcb39bf2da9ae9bb477286"
input_root = MODEL_ROOT.parent.resolve()
input_root.mkdir(parents=True, exist_ok=True)
bundle = input_root.parent / "cube555-inputs.zip"
def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()
if not bundle.is_file() or file_sha256(bundle) != INPUT_BUNDLE_SHA256:
    print("Downloading Cube555 inputs from GitHub", flush=True)
    partial = bundle.with_suffix(".partial")
    with urllib.request.urlopen(INPUT_BUNDLE_URL, timeout=120) as response, partial.open("wb") as stream:
        shutil.copyfileobj(response, stream)
    if file_sha256(partial) != INPUT_BUNDLE_SHA256:
        raise RuntimeError("Cube555 input ZIP SHA-256 mismatch")
    partial.replace(bundle)
with zipfile.ZipFile(bundle) as archive:
    for member in archive.infolist():
        if not (input_root / member.filename).resolve().is_relative_to(input_root):
            raise RuntimeError("Unsafe input ZIP member")
    archive.extractall(input_root)
input_manifest = json.loads((input_root / "manifest.json").read_text())
for name, entry in input_manifest.items():
    path = (input_root / name).resolve()
    if not path.is_relative_to(input_root):
        raise RuntimeError("Unsafe input manifest path")
    if path.stat().st_size != entry["bytes"] or file_sha256(path) != entry["sha256"]:
        raise RuntimeError(f"Cube555 input verification failed: {name}")
print("GitHub inputs verified:", len(input_manifest), "files", flush=True)

for path in (CHECKPOINT_PATH, RESMLP_CHECKPOINT_PATH, LAYOUT_PATH,
             MODEL_ROOT / "puzzle_info.json", COMPETITION_ROOT / "test.csv",
             COMPETITION_ROOT / "sample_submission.csv"):
    if not path.is_file():
        raise FileNotFoundError(f"Missing input: {path}. Check the GitHub download or edit its path.")


In [ ]:
import json, subprocess, sys, time
import torch
names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
assert len(names) == 2 and all("T4" in name for name in names), names
print("GPUs:", names, "torch:", torch.__version__, flush=True)
repo = Path("/tmp/cube555_native_repo")
if not repo.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
        "https://github.com/TryDotAtwo/MultiGPUBeamSearch.git", str(repo)], check=True)
subprocess.run(["git", "fetch", "--depth", "1", "origin", SOLVER_COMMIT], cwd=repo, check=True)
subprocess.run(["git", "checkout", "--detach", SOLVER_COMMIT], cwd=repo, check=True)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
assert actual == SOLVER_COMMIT, (actual, SOLVER_COMMIT)
sys.path.insert(0, str(repo))
assets = MODEL_ROOT
competition = COMPETITION_ROOT
output = Path("/kaggle/working") / ("cube555_" + time.strftime("%Y%m%d_%H%M%S"))
output.mkdir()
(output / "provenance.json").write_text(json.dumps(dict(solver_commit=actual,
    torch=torch.__version__, gpus=names, smoke=SMOKE_TEST,
    input_bundle_url=INPUT_BUNDLE_URL, input_bundle_sha256=INPUT_BUNDLE_SHA256,
    input_files=input_manifest), indent=2))


In [ ]:
from tools.cube555.smoke import make_fixture, validate_cuda
if SMOKE_TEST:
    validate_cuda(assets, output / "cuda_parity.json")
    competition = Path("/tmp/cube555_smoke_fixture")
    make_fixture(assets, competition)
    pids, beam, depth, radius = [0, 1, 2, 3], 4096, 6, 0
else:
    pids, beam, depth, radius = PUZZLE_IDS, BEAM_WIDTH, MAX_DEPTH, TOUCH_BFS_RADIUS
command = [sys.executable, "-u", "-m", "tools.cube555.run",
    "--assets", str(assets), "--competition", str(competition),
    "--checkpoint", str(CHECKPOINT_PATH), "--mlp-checkpoint", str(RESMLP_CHECKPOINT_PATH),
    "--layout", str(LAYOUT_PATH),
    "--output", str(output / "run"), "--pids", *map(str, pids),
    "--beam", str(beam), "--depth", str(depth), "--touch-radius", str(radius),
    "--transformer-weight", str(TRANSFORMER_WEIGHT)]
command += ["--b-micro", str(B_MICRO), "--model-micro", str(MODEL_MICRO)]
if not SMOKE_TEST:
    command += ["--reflect-mode", REFLECT_MODE, "--solution-mode", SOLUTION_MODE,
        "--collect-extra-depths", str(COLLECT_EXTRA_DEPTHS),
        "--max-collected-solutions", str(MAX_COLLECTED_SOLUTIONS)]
    if REFLECT_SOURCE_CSV is not None:
        command += ["--reflect-source-csv", str(REFLECT_SOURCE_CSV)]
if PUBLISH_RESULTS and not SMOKE_TEST:
    import hashlib
    notebook_path = Path("/kaggle/working/__notebook__.ipynb")
    if not notebook_path.exists():
        notebook_path = Path("__notebook__.ipynb")
    try:
        if not notebook_path.exists():
            raise FileNotFoundError("Cannot publish without the running notebook source")
        source = json.loads(notebook_path.read_text())
        source_cells = [{"cell_type": c["cell_type"], "source": c["source"]} for c in source["cells"]]
        source_hash = hashlib.sha256(json.dumps(source_cells, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
        publication = dict(competition="cayley-py-555-cube", kaggle_owner=KAGGLE_OWNER,
            kaggle_slug=KAGGLE_SLUG, kaggle_version=KAGGLE_VERSION,
            kaggle_username=KAGGLE_OWNER, solver_commit=SOLVER_COMMIT,
            kaggle_notebook_sha256=source_hash)
        # Hash identifies actual running cell sources, including the edited configuration.
        (output / "publication.json").write_text(json.dumps(publication, indent=2))
        command += ["--publish", "--publication-json", str(output / "publication.json")]
    except Exception as error:
        message = f"Publication setup failed: {type(error).__name__}: {error}"
        (output / "publication_setup_error.json").write_text(json.dumps(dict(error=message)))
        print(message + "; continuing search with local result files.", flush=True)
print("Run:", command, flush=True)
with (output / "launcher.log").open("w") as log:
    process = subprocess.Popen(command, cwd=repo, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()
    code = process.wait()
assert code == 0, f"native launcher exited {code}; inspect {output}"


In [ ]:
import pandas as pd
summary = json.loads((output / "run/run_summary.json").read_text())
print(json.dumps(summary, indent=2))
submission = output / "run/submission.csv"
display(pd.read_csv(submission).head(20))
if SMOKE_TEST:
    for pid in range(4):
        solutions = pd.read_csv(output / f"run/puzzle-{pid}/solutions/solutions.csv")
        assert len(solutions) > 0, f"smoke puzzle {pid} not solved"
        assert solutions["valid"].all(), f"smoke puzzle {pid} failed replay"
    print("PASS: four legal Cube555 scrambles solved and replayed by two native ranks")
